## Limpeza do tracking mobile: raw para cleaned
Lê `workspace.raw.tracking_events`, converte os tipos, aplica 10 tratamentos de qualidade e salva em `workspace.cleaned.tracking_events`, junto com a tabela `cleaning_log_mongo`, que mostra o efeito de cada tratamento. Este notebook não depende do Neon, então pode rodar em paralelo com a `apollo_cleaned`.

#### Configuração

##### Imports e camadas

In [0]:
import re
from pyspark.sql import functions as F
from pyspark.sql.types import StringType
from pyspark.sql.window import Window

CATALOG = "workspace"
TABLE = "tracking_events"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.cleaned")

##### Configurações dos tratamentos
Tudo que depende do domínio fica aqui. Quando aparecerem erros novos no tracking, é só ajustar esta célula.

In [0]:
FAKE_NULLS = ["", "s/n", "n/a", "na", "n.a.", "null", "none", "nan", "nil", "-", "--", "?", "undefined",
              "sem informacao", "sem informação", "nao informado", "não informado", "#n/a"]

LOWER_COLUMNS = ["event_name", "platform", "source", "properties_exit_reason", "properties_input_method"]
UPPER_COLUMNS = ["properties_method"]

TYPES = {
    "company_id": "bigint",
    "schema_version": "int",
    "timestamp": "timestamp",
    "received_at": "timestamp",
    "properties_duration_ms": "double",
    "properties_http_status": "int",
}

UUID_COLUMNS = ["event_id", "session_id", "user_id"]

RANGES = {
    "properties_duration_ms": (0, 86_400_000),  
    "properties_http_status": (100, 599),
    "schema_version": (1, 1000),
}

MIN_TIMESTAMP = "2020-01-01"                   
TIME_COLUMNS = ["timestamp", "received_at"]

REQUIRED_KEYS = ["event_id", "event_name", "timestamp", "session_id"]
IGNORE_IN_DUPLICATES = ["mongo_id", "ingested_at"]  

#### Tratamentos de texto

##### 1. Espaços extras

In [0]:
def text_columns(df):
    return [f.name for f in df.schema.fields if isinstance(f.dataType, StringType)]

def trim_texts(df):
    for column in text_columns(df):
        df = df.withColumn(column, F.trim(F.regexp_replace(F.col(column), r"\s+", " ")))
    return df

##### 2. Nulo escrito como texto

In [0]:
def fake_null_to_null(df):
    for column in text_columns(df):
        df = df.withColumn(column, F.when(F.lower(F.col(column)).isin(FAKE_NULLS), None).otherwise(F.col(column)))
    return df

##### 3. Caixa das categorias

In [0]:
def normalize_categories(df):
    for column in LOWER_COLUMNS:
        if column in df.columns:
            df = df.withColumn(column, F.lower(F.col(column)))
    for column in UPPER_COLUMNS:
        if column in df.columns:
            df = df.withColumn(column, F.upper(F.col(column)))
    return df

##### 4. Identificador com formato inválido

In [0]:
UUID_REGEX = r"^[0-9a-fA-F]{8}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{12}$"

def invalid_uuid_to_null(df):
    for column in UUID_COLUMNS:
        if column in df.columns:
            df = df.withColumn(column, F.when(F.col(column).rlike(UUID_REGEX), F.col(column)))
    return df

#### Tratamentos de tipo, número e data

##### 5. Conversão de tipos

In [0]:
def cast_types(df):
    for column, dtype in TYPES.items():
        if column in df.columns:
            if dtype in ("int", "bigint"):
                expr = f"try_cast(try_cast(`{column}` as double) as {dtype})"   # aceita "200" e "200.0"
            else:
                expr = f"try_cast(`{column}` as {dtype})"
            df = df.withColumn(column, F.expr(expr))
    return df

##### 6. NaN e infinito

In [0]:
def nan_to_null(df):
    for field in df.schema.fields:
        if field.dataType.simpleString() == "double":
            column = field.name
            bad = F.isnan(column) | F.col(column).isin(float("inf"), float("-inf"))
            df = df.withColumn(column, F.when(bad, None).otherwise(F.col(column)))
    return df

##### 7. Fora da faixa

In [0]:
def out_of_range_to_null(df):
    for column, (low, high) in RANGES.items():
        if column in df.columns:
            df = df.withColumn(column, F.when(F.col(column).between(low, high), F.col(column)))
    return df

##### 8. Data impossível

In [0]:
def bad_timestamps_to_null(df):
    limit = F.current_timestamp() + F.expr("INTERVAL 1 DAY")
    for column in TIME_COLUMNS:
        if column in df.columns:
            invalid = (F.col(column) > limit) | (F.col(column) < F.lit(MIN_TIMESTAMP).cast("timestamp"))
            df = df.withColumn(column, F.when(invalid, None).otherwise(F.col(column)))
    return df

#### Tratamentos de chave e duplicidade

##### 9. Evento sem campo obrigatório

In [0]:
def drop_null_keys(df):
    return df.dropna(subset=[c for c in REQUIRED_KEYS if c in df.columns])

##### 10. Evento duplicado

In [0]:
def drop_duplicates(df):
    content = [c for c in df.columns if c not in IGNORE_IN_DUPLICATES]
    df = df.dropDuplicates(content)
    w = Window.partitionBy("event_id").orderBy(F.col("received_at").desc_nulls_last(), F.col("ingested_at").desc_nulls_last())
    return df.withColumn("_rn", F.row_number().over(w)).filter(F.col("_rn") == 1).drop("_rn")

#### Salvar na cleaned

##### Ordem dos tratamentos
A ordem importa: primeiro padroniza o texto, depois converte e zera o que é inválido, e só então remove linhas.

In [0]:
# (nome, função, efeito): "valor" = transforma valores em nulo, "linha" = remove linhas, None = só padroniza
STEPS = [
    ("1 espacos extras",           trim_texts, None),
    ("2 nulo escrito como texto",  fake_null_to_null, "valor"),
    ("3 caixa das categorias",     normalize_categories, None),
    ("4 identificador invalido",   invalid_uuid_to_null, "valor"),
    ("5 conversao de tipos",       cast_types, "valor"),
    ("6 nan e infinito",           nan_to_null, "valor"),
    ("7 fora da faixa",            out_of_range_to_null, "valor"),
    ("8 data impossivel",          bad_timestamps_to_null, "valor"),
    ("9 campo obrigatorio nulo",   drop_null_keys, "linha"),
    ("10 evento duplicado",        drop_duplicates, "linha"),
]

##### Medir o efeito de cada tratamento

In [0]:
def snapshot(df):
    nulls = sum((F.col(c).isNull().cast("int") for c in df.columns), F.lit(0))
    row = df.agg(F.count("*").alias("linhas"), F.sum(nulls).alias("nulos")).first()
    return row["linhas"], int(row["nulos"] or 0)

def clean_table(df, log):
    rows, nulls = snapshot(df)
    for name, step, effect in STEPS:
        new_df = step(df)
        if effect:
            new_rows, new_nulls = snapshot(new_df)
            removed = rows - new_rows if effect == "linha" else 0
            created = new_nulls - nulls if effect == "valor" else 0
            log.append((TABLE, name, removed, created))
            rows, nulls = new_rows, new_nulls
        df = new_df
    return df

##### Rodar o tratamento e salvar

In [0]:
log = []
raw_df = spark.table(f"{CATALOG}.raw.{TABLE}")
cleaned_df = clean_table(raw_df, log)
cleaned_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.cleaned.{TABLE}")
print(TABLE, "ok:", raw_df.count() - spark.table(f"{CATALOG}.cleaned.{TABLE}").count(), "linhas removidas")

##### Salvar o log dos tratamentos

In [0]:
log_df = (spark.createDataFrame(log, "tabela string, tratamento string, linhas_removidas long, nulos_criados long")
          .filter("linhas_removidas != 0 OR nulos_criados != 0")
          .withColumn("executado_em", F.current_timestamp()))
log_df.write.mode("overwrite").saveAsTable(f"{CATALOG}.cleaned.cleaning_log_mongo")
display(log_df.orderBy("tratamento"))

#### Conferir

In [0]:
display(spark.table(f"{CATALOG}.cleaned.{TABLE}").limit(20))
display(spark.table(f"{CATALOG}.cleaned.{TABLE}").groupBy("event_name").count().orderBy("count", ascending=False))